In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

### Bronze layer for Incremental Fact

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","orders","Data_source")

catalog=dbutils.widgets.get("catalog")
data_source=dbutils.widgets.get("data_source")

base_path="s3://sportsbar-dp-sh/orders"
landing_path=f"{base_path}/landing"
processed_path=f"{base_path}/processd"

print("[Info  ]:landing path -->",landing_path)

print("[Info  ]:processed path -->",processed_path)

bronze_table=f"{catalog}.{bronze_schema}.{data_source}"
silver_table=f"{catalog}.{silver_schema}.{data_source}"
gold_table=f"{catalog}.{gold_schema}.sb_fact_{data_source}"

df_bronze=spark.read\
    .option("inferSchema",True)\
    .option("header",True)\
    .csv(f"{landing_path}")

df_bronze=(df_bronze.withColumn("ingestime",F.current_timestamp())
           .withColumn("file_name",F.col("_metadata.file_name"))
           .withColumn("file_size",F.col("_metadata.file_size"))
)

df_bronze.write.mode("delta")\
    .mode("append")\
    .option("delta.enableChangeDataFeed",True)\
    .saveAsTable(f"{bronze_table}")

### Staging table to process just the arrived incremnetal data

In [0]:
df_bronze.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", True) \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.staging_{data_source}")

In [0]:
files=dbutils.fs.ls(f"{landing_path}")
for file_info in files:
    dbutils.fs.mv(file_info.path,f"{processed_path}/file_info.name")



### Silver Layer Processing

In [0]:
df_silver=spark.table(f"{catalog}.{bronze_schema}.staging_{data_source}")


# 1. Keep only rows where order_qty and order_id is present

df_silver=df_silver.filter((F.col("order_qty").isNotNull()) & (F.col("order_id").isNotNull()))

df_silver=df_silver.filter(F.col("order_placement_date").isNotNull())

# 2. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"

df_silver=df_silver.withColumn("order_placement_date",F.regexp_replace(F.col("order_placement_date"),r"^[A-Za-z]+,\s*",""))

# 2. Clean customer_id → keep numeric, else set to 999999

df_silver=df_silver.withColumn("customer_id",F.when(F.col("customer_id").rlike(r"^[0-9]+$"),F.col("customer_id")).otherwise("999999").cast("string"))

# 4. Parse order_placement_date using multiple possible formats

df_silver=df_silver.withColumn(
    "order_placement_date",F.coalesce
    (
    F.try_to_date("order_placement_date","yyyy/MM/dd"),
    F.try_to_date("order_placement_date","dd-MM-yyyy"),
    F.try_to_date("order_placement_date","MM-dd-yyyy"),
    F.try_to_date("order_placement_date","MMMM dd, yyyy"),
    )
)
# 5. Drop duplicates

df_silver=df_silver.dropDuplicates(["order_id","order_placement_date","customer_id","product_id","order_qty"])

# 5. convert product id to string

df_silver = df_silver.withColumn('product_id', F.col('product_id').cast('string'))


In [0]:
df_silver.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

Join with Products

In [0]:
df_products=spark.table(f"{catalog}.{silver_schema}.products")

df_joined=df_silver.join(df_products, on="product_id",how="inner").select(df_silver["*"],df_products["product_code"])

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta")\
        .mode("overwrite")\
        .option("delta.enableChangeDataFeed",True)\
        .saveAsTable(f"{silver_table}")
else:
    silver_table=DeltaTable.forName(spark,silver_table)
    silver_table.alias("silver").merge(
        df_joined.alias("bronze"),
        "silver.order_id=bronze.order_id AND silver.order_placement_date=bronze.order_placement_date AND silver.customer_id=bronze.customer_id AND silver.product_id=bronze.product_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()




### Staging table to process just the arrived incremenal data

In [0]:
# stagging for incremental data

df_joined.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.staging_{data_source}")

### Gold Processing Layer

In [0]:
df_gold=spark.table(f"{catalog}.{silver_schema}.staging_{data_source}")

df_gold=df_gold.select(F.col("order_id"),F.col("order_placement_date").alias("date"),F.col("customer_id").alias("customer_code"),F.col("product_code"),
                       F.col("product_id"),F.col("order_qty").alias("sold_quantity"))

In [0]:
if not spark.catalog.tableExists(gold_table):
    print("Creating new table")

    df_gold.write \
        .format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .option("mergeSchema", "true") \
        .mode("overwrite") \
        .saveAsTable(gold_table)

else:
    print("Table already exists. Merging data.")

    gold_delta = DeltaTable.forName(spark, gold_table)

    gold_delta.alias("target") \
        .merge(
            df_gold.alias("source"),
            """
            target.date = source.date
            AND target.order_id = source.order_id
            AND target.product_code = source.product_code
            AND target.customer_code = source.customer_code
            """
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

## Merging with Parent company

- Note: We want data for monthly level but child data is on daily level

Incremental Load

In [0]:
# df_child = your incremental daily rows

df_child =  spark.sql(f"SELECT order_placement_date as date FROM {catalog}.{silver_schema}.staging_{data_source}")

incremental_month_df = df_child.select(
    F.trunc("date", "MM").alias("start_month")
).distinct()

incremental_month_df.show()

incremental_month_df.createOrReplaceTempView("incremental_months")

In [0]:
monthly_table = spark.sql(f"""
    SELECT date, product_code, customer_code, sold_quantity
    FROM {catalog}.{gold_schema}.sb_fact_orders sbf
    INNER JOIN incremental_months m
        ON trunc(sbf.date, 'MM') = m.start_month
""")

print("Total Rows: ", monthly_table.count())
monthly_table.show(10)

In [0]:
monthly_table.select('date').distinct().orderBy('date').show()

In [0]:
df_monthly_recalc = (
    monthly_table
    .withColumn("month_start", F.trunc("date", "MM"))
    .groupBy("month_start", "product_code", "customer_code")
    .agg(F.sum("sold_quantity").alias("sold_quantity"))
    .withColumnRenamed("month_start", "date")   # month_start → date = first of month
)

df_monthly_recalc.show(10, truncate=False)

In [0]:
df_monthly_recalc.count()

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly_recalc.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

### Cleanup

In [0]:
%sql
DROP TABLE fmcg.bronze.staging_orders;

In [0]:
%sql
DROP TABLE fmcg.silver.staging_orders;